## Dependencies

In [2]:
import numpy as np
import pandas as pd
import random

import ast
import re
import GEOparse
from itertools import islice
from bioservices import KEGG
from collections import Counter

Creating directory /root/.config/bioservices 


## Data Processing

In [ ]:
y = pd.read_csv('original/actual.csv')
print(y.shape)
y.head()

# acute myeloid leukemia (AML) and acute lymphoblastic leukemia (ALL)
# training 38 samples
# testing 34 samples

(72, 2)


,patient,cancer
0,1,ALL
1,2,ALL
2,3,ALL
3,4,ALL
4,5,ALL


In [4]:
y['cancer'].value_counts()

,count
cancer,
ALL,47
AML,25


In [ ]:
# Recode label to numeric
y = y.replace({'ALL':0,'AML':1})
labels = ['ALL', 'AML'] 

/tmp/ipython-input-1607318789.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = y.replace({'ALL':0,'AML':1})


In [ ]:
# Import training data
df_train = pd.read_csv('original/data_set_ALL_AML_train.csv')
print(df_train.shape)

# Import testing data
df_test = pd.read_csv('original/data_set_ALL_AML_independent.csv')
print(df_test.shape)

(7129, 78)
(7129, 70)


In [7]:
df_train.head()

,Gene Description,Gene Accession Number,1,call,2,call.1,3,call.2,4,call.3,...,29,call.33,30,call.34,31,call.35,32,call.36,33,call.37
0,AFFX-BioB-5_at (endogenous control),AFFX-BioB-5_at,-214,A,-139,A,-76,A,-135,A,...,15,A,-318,A,-32,A,-124,A,-135,A
1,AFFX-BioB-M_at (endogenous control),AFFX-BioB-M_at,-153,A,-73,A,-49,A,-114,A,...,-114,A,-192,A,-49,A,-79,A,-186,A
2,AFFX-BioB-3_at (endogenous control),AFFX-BioB-3_at,-58,A,-1,A,-307,A,265,A,...,2,A,-95,A,49,A,-37,A,-70,A
3,AFFX-BioC-5_at (endogenous control),AFFX-BioC-5_at,88,A,283,A,309,A,12,A,...,193,A,312,A,230,P,330,A,337,A
4,AFFX-BioC-3_at (endogenous control),AFFX-BioC-3_at,-295,A,-264,A,-376,A,-419,A,...,-51,A,-139,A,-367,A,-188,A,-407,A


In [8]:
# Remove "call" columns from training and testing data
train_to_keep = [col for col in df_train.columns if "call" not in col]
test_to_keep = [col for col in df_test.columns if "call" not in col]

X_train_tr = df_train[train_to_keep]
X_test_tr = df_test[test_to_keep]

In [9]:
# Reorder columns in numeric order
train_columns_titles = ['Gene Description', 'Gene Accession Number', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10',
       '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25',
       '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38']

X_train_tr = X_train_tr.reindex(columns=train_columns_titles)

In [10]:
test_columns_titles = ['Gene Description', 'Gene Accession Number','39', '40', '41', '42', '43', '44', '45', '46',
       '47', '48', '49', '50', '51', '52', '53',  '54', '55', '56', '57', '58', '59',
       '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72']

X_test_tr = X_test_tr.reindex(columns=test_columns_titles)

In [11]:
# Transpose the columns and rows so that genes become features and each patient's observations occupies a single row
X_train = X_train_tr.T
X_test = X_test_tr.T

print(X_train.shape)
X_train.head()

(40, 7129)


,0,1,2,3,4,5,6,7,8,9,...,7119,7120,7121,7122,7123,7124,7125,7126,7127,7128
Gene Description,AFFX-BioB-5_at (endogenous control),AFFX-BioB-M_at (endogenous control),AFFX-BioB-3_at (endogenous control),AFFX-BioC-5_at (endogenous control),AFFX-BioC-3_at (endogenous control),AFFX-BioDn-5_at (endogenous control),AFFX-BioDn-3_at (endogenous control),AFFX-CreX-5_at (endogenous control),AFFX-CreX-3_at (endogenous control),AFFX-BioB-5_st (endogenous control),...,Transcription factor Stat5b (stat5b) mRNA,Breast epithelial antigen BA46 mRNA,GB DEF = Calcium/calmodulin-dependent protein ...,TUBULIN ALPHA-4 CHAIN,CYP4B1 Cytochrome P450; subfamily IVB; polypep...,PTGER3 Prostaglandin E receptor 3 (subtype EP3...,HMG2 High-mobility group (nonhistone chromosom...,RB1 Retinoblastoma 1 (including osteosarcoma),GB DEF = Glycophorin Sta (type A) exons 3 and ...,GB DEF = mRNA (clone 1A7)
Gene Accession Number,AFFX-BioB-5_at,AFFX-BioB-M_at,AFFX-BioB-3_at,AFFX-BioC-5_at,AFFX-BioC-3_at,AFFX-BioDn-5_at,AFFX-BioDn-3_at,AFFX-CreX-5_at,AFFX-CreX-3_at,AFFX-BioB-5_st,...,U48730_at,U58516_at,U73738_at,X06956_at,X16699_at,X83863_at,Z17240_at,L49218_f_at,M71243_f_at,Z78285_f_at
1,-214,-153,-58,88,-295,-558,199,-176,252,206,...,185,511,-125,389,-37,793,329,36,191,-37
2,-139,-73,-1,283,-264,-400,-330,-168,101,74,...,169,837,-36,442,-17,782,295,11,76,-14
3,-76,-49,-307,309,-376,-650,33,-367,206,-215,...,315,1199,33,168,52,1138,777,41,228,-41


In [12]:
# Clean up the column names for training and testing data
X_train.columns = X_train.iloc[1]
X_train = X_train.drop(["Gene Description", "Gene Accession Number"]).apply(pd.to_numeric)

# Clean up the column names for Testing data
X_test.columns = X_test.iloc[1]
X_test = X_test.drop(["Gene Description", "Gene Accession Number"]).apply(pd.to_numeric)

print(X_train.shape)
print(X_test.shape)
X_train.head()

(38, 7129)
(34, 7129)


Gene Accession Number,AFFX-BioB-5_at,AFFX-BioB-M_at,AFFX-BioB-3_at,AFFX-BioC-5_at,AFFX-BioC-3_at,AFFX-BioDn-5_at,AFFX-BioDn-3_at,AFFX-CreX-5_at,AFFX-CreX-3_at,AFFX-BioB-5_st,...,U48730_at,U58516_at,U73738_at,X06956_at,X16699_at,X83863_at,Z17240_at,L49218_f_at,M71243_f_at,Z78285_f_at
1,-214,-153,-58,88,-295,-558,199,-176,252,206,...,185,511,-125,389,-37,793,329,36,191,-37
2,-139,-73,-1,283,-264,-400,-330,-168,101,74,...,169,837,-36,442,-17,782,295,11,76,-14
3,-76,-49,-307,309,-376,-650,33,-367,206,-215,...,315,1199,33,168,52,1138,777,41,228,-41
4,-135,-114,265,12,-419,-585,158,-253,49,31,...,240,835,218,174,-110,627,170,-50,126,-91
5,-106,-125,-76,168,-230,-284,4,-122,70,252,...,156,649,57,504,-26,250,314,14,56,-25


In [13]:
# Combine X_train and X_test into one DataFrame
X_train_r = X_train.copy()
X_test_r  = X_test.copy()

# Combine train and test in the same order as y (train first, then test)
X_combined = pd.concat([X_train_r, X_test_r], ignore_index=True)
X_combined.insert(0, 'patient', y['patient'].values)

# Add y as last column
X_combined['label'] = y['cancer'].values

if 'patient' in X_combined.columns:
    X_combined_final = X_combined.drop(columns=['patient'])

# Save to CSV
X_combined_final.to_csv('original.csv', index=False)
print("Saved combined dataframe to 'original.csv'")


Saved combined dataframe to 'original.csv'


In [14]:
print(X_combined_final.shape)
X_combined_final.head()

(72, 7130)


Gene Accession Number,AFFX-BioB-5_at,AFFX-BioB-M_at,AFFX-BioB-3_at,AFFX-BioC-5_at,AFFX-BioC-3_at,AFFX-BioDn-5_at,AFFX-BioDn-3_at,AFFX-CreX-5_at,AFFX-CreX-3_at,AFFX-BioB-5_st,...,U58516_at,U73738_at,X06956_at,X16699_at,X83863_at,Z17240_at,L49218_f_at,M71243_f_at,Z78285_f_at,label
0,-214,-153,-58,88,-295,-558,199,-176,252,206,...,511,-125,389,-37,793,329,36,191,-37,0
1,-139,-73,-1,283,-264,-400,-330,-168,101,74,...,837,-36,442,-17,782,295,11,76,-14,0
2,-76,-49,-307,309,-376,-650,33,-367,206,-215,...,1199,33,168,52,1138,777,41,228,-41,0
3,-135,-114,265,12,-419,-585,158,-253,49,31,...,835,218,174,-110,627,170,-50,126,-91,0
4,-106,-125,-76,168,-230,-284,4,-122,70,252,...,649,57,504,-26,250,314,14,56,-25,0


## Probe to symbol

In [ ]:
def fetch_gpl_table(gpl_acc="GPL80", destdir="."):
    print(f"Downloading {gpl_acc} ...")
    gpl = GEOparse.get_GEO(geo=gpl_acc, destdir=destdir, silent=True)
    table = gpl.table
    print("GPL downloaded. Table shape:", table.shape)
    return table

def build_probe2symbol_from_table(table, probe_col, symbol_col):
    mapping = {}
    missing = 0
    for idx, row in table.iterrows():
        probe = str(row[probe_col]).strip()
        symbol_raw = row.get(symbol_col, "")
        if pd.isna(symbol_raw):
            mapping[probe] = None
            missing += 1
            continue
        symbol = str(symbol_raw).strip()
        # Clean symbol
        symbol = re.split(r"\s*///\s*|\s*//\s*|;", symbol)[0]
        tokens = re.split(r"[,\s\(\)]+", symbol)

        sym_candidate = None
        for t in tokens:
            if re.match(r"^[A-Za-z0-9\-\._]+$", t):
                if t.upper() in ("NA", "---"):
                    continue
                if re.search(r"[A-Za-z]", t):
                    sym_candidate = t
                    break
        if sym_candidate is None:
            sym_candidate = symbol  
        mapping[probe] = sym_candidate
    print(f"Built mapping for {len(mapping)} probes ({missing} missing symbols).")
    return mapping

In [16]:
gpl_table = fetch_gpl_table("GPL80")
probe_col = "ID"
symbol_col = "Gene Symbol"

GPL downloaded. Table shape: (7129, 16)


In [17]:
probe2symbol = build_probe2symbol_from_table(gpl_table, probe_col, symbol_col)

print("Example of probe2symbol: ")
example1 = dict(islice(probe2symbol.items(), 5))
example1

Built mapping for 7129 probes (701 missing symbols).
Example of probe2symbol: 


{'A28102_at': 'GABRA3',
 'AB000114_at': 'OMD',
 'AB000115_at': 'IFI44L',
 'AB000220_at': 'SEMA3C',
 'AB000381_s_at': 'GML'}

In [18]:
dataset_genes = list(set(probe2symbol.values()))
print("Number of unique symbols:", len(dataset_genes))

Number of unique symbols: 5546


In [19]:
def map_col2gene_df(df, probe2symbol, label_col):

    df = df.copy()

    # identify probe columns in the dataframe that exist in probe2symbol
    probes_in_df = [c for c in df.columns if c in probe2symbol]
    # drop probes with missing symbol mapping
    probes_with_symbol = [p for p in probes_in_df if probe2symbol.get(p) is not None]

    # compose subset columns to keep
    keep_cols = probes_with_symbol[:]
    keep_cols.append(label_col)

    df = df.loc[:, keep_cols]

    # rename probes to gene symbols
    rename_map = {probe: probe2symbol[probe] for probe in probes_with_symbol}
    df = df.rename(columns=rename_map)

    # collapse probes that map to the same symbol by mean
    expr = df.drop(columns=[label_col])
    cancer_series = df[label_col]
    expr = expr.groupby(expr.columns, axis=1).mean()

    # reattach cancer column
    final_df = pd.concat([expr, cancer_series.reset_index(drop=True)], axis=1)
    final_df = final_df.rename(columns={final_df.columns[-1]: label_col})

    return final_df

In [20]:
gene_symbol_df = map_col2gene_df(X_combined_final, probe2symbol, label_col="label")

print(gene_symbol_df.shape)
gene_symbol_df.head(5)

(72, 5546)


/tmp/ipython-input-566648092.py:23: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  expr = expr.groupby(expr.columns, axis=1).mean()


,A2M,AADAC,AAMP,AANAT,AARS,ABAT,ABCA2,ABCA3,ABCA4,ABCB1,...,ZNF85,ZNF91,ZNHIT3,ZP2,ZPBP,ZRSR2,ZSCAN26,ZXDA,ZYX,label
0,184.0,194.0,1228.0,2030.0,1530.0,358.333333,-555.0,-10.0,921.0,153.5,...,-18.0,6732.5,534.0,279.0,-209.0,-540.0,70.0,88.0,298.0,0
1,4.0,119.0,1054.0,1314.0,536.0,191.333333,-603.0,0.0,49.0,1.0,...,0.0,5299.5,566.0,250.0,-171.0,-509.0,120.0,180.0,307.0,0
2,274.0,285.0,1927.0,2389.0,1481.0,383.000000,-638.0,127.0,-118.0,119.5,...,10.0,6864.5,533.0,278.0,-195.0,-1319.0,181.0,42.0,309.0,0
3,144.0,220.0,1095.0,2047.0,1431.0,374.000000,-660.0,-34.0,105.0,87.0,...,-6.0,4614.0,584.0,472.0,-191.0,-798.0,134.0,205.0,693.0,0
4,25.0,32.0,1197.0,981.0,1479.0,111.000000,-652.0,-21.0,-101.0,50.5,...,23.0,2765.0,781.0,256.0,-34.0,-186.0,59.0,44.0,713.0,0


In [21]:
gene_symbol_df.to_csv("original_mapped.csv", index=True)
print("saved csv")

saved csv


In [22]:
gene_symbols = {c for c in gene_symbol_df.columns if c != "label"}

## Get genes from KEGG cancer pathways

In [ ]:
unique_intersect_gene = set()

In [ ]:
def fetch_kegg_aml_symbols(pathway):
    k = KEGG()
    entry = k.get(pathway)
    parsed = k.parse(entry)
    symbols = set()
    if 'GENE' in parsed:
        for entrez, desc in parsed['GENE'].items():
            m = re.match(r"^([A-Za-z0-9\-\._]+)", desc)
            if m:
                symbols.add(m.group(1).upper().strip())
    return symbols

def add_to_set(list):
    for item in list:
      if item not in unique_intersect_gene:
        unique_intersect_gene.add(item)
      pass

def get_intersect_genes(gene_symbols, pathway):
    symbols_in_pathway = fetch_kegg_aml_symbols(pathway)
    print(f'Number of symbols in {pathway}: {len(symbols_in_pathway)}')

    intersected_genes = sorted(list(gene_symbols.intersection(symbols_in_pathway)))
    print("Matched genes:", len(intersected_genes))

    add_to_set(intersected_genes)
    print("Number of intersected gene: ", len(unique_intersect_gene))


In [ ]:
kegg_pathway_list = [
    'hsa05200',
    'hsa05205',
    'hsa05206',
    'hsa05202',
    'hsa05223',
    'hsa05230',
    'hsa05222',
    'hsa05224',
    'hsa05215',
    'hsa05213',
    'hsa05226',
    'hsa05210',
    'hsa05212',
    'hsa05216',
    'hsa05219',
    'hsa05231',
    'hsa05235',
    'hsa05217',
    'hsa05208',
    'hsa05203',
    'hsa05169',
    'hsa05211',
    'hsa05204',
    'hsa05225',
    'hsa05207',
    'hsa05218',
    'hsa05214',
    'hsa05165',
    'hsa04371',
    'hsa03460',
    'hsa03320'
]

In [ ]:
for i in kegg_pathway_list:
  get_intersect_genes(gene_symbols, i)

Number of symbols in hsa05200: 533
Matched genes: 369
Number of intersected gene:  369
Number of symbols in hsa05205: 204
Matched genes: 142
Number of intersected gene:  434
Number of symbols in hsa05206: 320
Matched genes: 113
Number of intersected gene:  476
Number of symbols in hsa05202: 201
Matched genes: 127
Number of intersected gene:  552
Number of symbols in hsa05223: 73
Matched genes: 57
Number of intersected gene:  555
Number of symbols in hsa05230: 71
Matched genes: 56
Number of intersected gene:  579
Number of symbols in hsa05222: 93
Matched genes: 72
Number of intersected gene:  579
Number of symbols in hsa05224: 148
Matched genes: 84
Number of intersected gene:  582
Number of symbols in hsa05215: 106
Matched genes: 81
Number of intersected gene:  594
Number of symbols in hsa05213: 59
Matched genes: 44
Number of intersected gene:  595
Number of symbols in hsa05226: 150
Matched genes: 91
Number of intersected gene:  598
Number of symbols in hsa05210: 87
Matched genes: 69
Nu

In [ ]:
with open("kegg_intersect.txt", "w") as f:
  f.write(repr(unique_intersect_gene))

## Extract subset from intersected genes

In [24]:
df = pd.read_csv('original_mapped.csv')
df.head(5)

,Unnamed: 0,A2M,AADAC,AAMP,AANAT,AARS,ABAT,ABCA2,ABCA3,ABCA4,...,ZNF85,ZNF91,ZNHIT3,ZP2,ZPBP,ZRSR2,ZSCAN26,ZXDA,ZYX,label
0,0,184.0,194.0,1228.0,2030.0,1530.0,358.333333,-555.0,-10.0,921.0,...,-18.0,6732.5,534.0,279.0,-209.0,-540.0,70.0,88.0,298.0,0
1,1,4.0,119.0,1054.0,1314.0,536.0,191.333333,-603.0,0.0,49.0,...,0.0,5299.5,566.0,250.0,-171.0,-509.0,120.0,180.0,307.0,0
2,2,274.0,285.0,1927.0,2389.0,1481.0,383.000000,-638.0,127.0,-118.0,...,10.0,6864.5,533.0,278.0,-195.0,-1319.0,181.0,42.0,309.0,0
3,3,144.0,220.0,1095.0,2047.0,1431.0,374.000000,-660.0,-34.0,105.0,...,-6.0,4614.0,584.0,472.0,-191.0,-798.0,134.0,205.0,693.0,0
4,4,25.0,32.0,1197.0,981.0,1479.0,111.000000,-652.0,-21.0,-101.0,...,23.0,2765.0,781.0,256.0,-34.0,-186.0,59.0,44.0,713.0,0


In [23]:
with open("kegg_intersect.txt") as f:
    initial_intersect = eval(f.read())

print(len(initial_intersect))
print(list(initial_intersect)[:10])

1027
['PFKM', 'NEDD4', 'PRKCA', 'MAP2K3', 'ATF4', 'IL12RB1', 'SP1', 'CACNA1S', 'BLM', 'PRKAG1']


In [25]:
df.columns = [str(c).strip() for c in df.columns]
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

In [26]:
df.head()

,A2M,AADAC,AAMP,AANAT,AARS,ABAT,ABCA2,ABCA3,ABCA4,ABCB1,...,ZNF85,ZNF91,ZNHIT3,ZP2,ZPBP,ZRSR2,ZSCAN26,ZXDA,ZYX,label
0,184.0,194.0,1228.0,2030.0,1530.0,358.333333,-555.0,-10.0,921.0,153.5,...,-18.0,6732.5,534.0,279.0,-209.0,-540.0,70.0,88.0,298.0,0
1,4.0,119.0,1054.0,1314.0,536.0,191.333333,-603.0,0.0,49.0,1.0,...,0.0,5299.5,566.0,250.0,-171.0,-509.0,120.0,180.0,307.0,0
2,274.0,285.0,1927.0,2389.0,1481.0,383.000000,-638.0,127.0,-118.0,119.5,...,10.0,6864.5,533.0,278.0,-195.0,-1319.0,181.0,42.0,309.0,0
3,144.0,220.0,1095.0,2047.0,1431.0,374.000000,-660.0,-34.0,105.0,87.0,...,-6.0,4614.0,584.0,472.0,-191.0,-798.0,134.0,205.0,693.0,0
4,25.0,32.0,1197.0,981.0,1479.0,111.000000,-652.0,-21.0,-101.0,50.5,...,23.0,2765.0,781.0,256.0,-34.0,-186.0,59.0,44.0,713.0,0


In [27]:
# keep only intersected genes that are present in the df
genes_present = [g for g in initial_intersect if g in df.columns]
missing_genes = [g for g in initial_intersect if g not in df.columns]
cols_to_keep = genes_present + ["label"]

print(f"Intersected genes found in dataframe: {len(genes_present)}")

Intersected genes found in dataframe: 1027


In [28]:
reduced_df = df.loc[:, cols_to_keep].copy()

print("Reduced dataframe shape (rows x cols):", reduced_df.shape)
print("Label distribution in reduced set:")
print(reduced_df["label"].value_counts())

Reduced dataframe shape (rows x cols): (72, 1028)
Label distribution in reduced set:
label
0    47
1    25
Name: count, dtype: int64


In [29]:
reduced_df.head()

,PFKM,NEDD4,PRKCA,MAP2K3,ATF4,IL12RB1,SP1,CACNA1S,BLM,PRKAG1,...,LUM,ATP6V1C1,PIK3R4,SMARCD3,ATF2,CYP1A1,KIT,LPAR4,NDUFV2,label
0,63.0,9.0,331.0,319.0,1095.0,-262.0,367.0,-637.0,-56.0,997.0,...,-3.0,128.0,233.0,-1448.0,321.0,35.0,74.0,329.0,704.0,0
1,-26.0,27.0,224.0,511.0,5457.0,-668.0,211.0,-438.0,2.0,863.0,...,-43.0,100.0,226.0,-1049.0,59.0,27.0,-4.0,357.0,4107.0,0
2,-114.0,114.0,624.0,246.0,2736.0,-547.0,410.0,-691.0,-375.0,1367.0,...,21.0,142.0,142.0,-1489.0,322.0,195.0,114.0,559.0,1010.0,0
3,-158.0,28.0,418.0,-25.0,1517.0,-706.0,366.0,-617.0,181.0,1039.0,...,-2.0,220.0,181.0,-1117.0,385.0,-41.0,-15.0,183.0,905.0,0
4,230.0,19.0,242.0,779.0,2939.0,-256.0,137.0,-15.0,97.0,1048.0,...,-20.0,167.0,226.0,-820.0,196.0,-5.0,12.0,243.0,2204.0,0


In [30]:
reduced_df.to_csv("reduced.csv", index=False)
print(f"Saved reduced dataframe")

Saved reduced dataframe
